# 4.2 Guided Practical: Simulating CLL & Drug Responses

In this guided practical, we assemble and run the full **NF-κB Tri-Model with Shared TAK1** live in Julia to compare:

1. **M-CLL + CpG** — Mutated CLL (low basal BCR signalling) stimulated with a TLR/CpG pulse
2. **U-CLL + CpG** — Unmutated CLL (high chronic basal BCR signalling) stimulated with the same TLR/CpG pulse
3. **U-CLL + CpG + IKK inhibitor** — Treating U-CLL with a targeted drug that accelerates IKK deactivation (`k1_IKKdeactivation-linking` × 10)

Click **⚡ Activate Live Editable Julia Cells** above to launch your private Julia workspace, then press `Shift + Enter` through each cell!

## The NF-κB Tri-Model Diagram

Every reaction in this diagram—across **TLR**, **BCR**, **NF-κB**, and the shared **TAK1 / IKK / A20** linking hub—is stored in modular CSV spreadsheets inside `/home/jovyan/work/NFkB/moduleDefinitionFiles/`.

![Tri-Model with Shared TAK1](../images/triModelA20Diagram.png)

## Step 1: Load Packages & Workspace Files

Run the cell below to move into the `NFkB/` model directory, configure the CSV-to-Julia paths, and load our Julia packages (`DifferentialEquations`, `Plots`, `CSV`, `DataFrames`).

In [ ]:
# Point Julia to the pre-installed NF-kB Tri-Model workspace
if isdir("/home/jovyan/work/NFkB")
    cd("/home/jovyan/work/NFkB")
elseif isdir("NFkB")
    cd("NFkB")
end
@assert isfile("combineModels.jl") "NFkB model files not found in $(pwd())."

# File locations for the modular CSV-to-Julia pipeline
locationOfCSV2Julia         = "csv2Julia/csv2model-multiscale.py"
moduleDefinitionFilesFolder = "moduleDefinitionFiles/"
delay                       = false

# Load scientific computing & plotting packages
using DifferentialEquations
using Plots
using CSV
using DataFrames

println("Loaded packages in: ", pwd())

Loaded packages in: /home/jovyan/work/NFkB

## Step 2: Combine Modules & Generate the 140-Species ODE Model

Here we combine the **BCR**, **TLR**, and **NF-κB** modules (automatically deduplicating reactions handled by the `linking` module) and run `csv2model-multiscale.py` to generate our complete Julia ODE system (`distributedModelFiles/odeModel.jl`, `variableNames.jl`, and `scanIncludes.jl`).

In [ ]:
include("combineModels.jl")
combinedModelLocation = combineModels(["BCR", "TLR", "NFkB"])

include("generateModel.jl")

## Step 3: Load Signalling Inputs & Simulation Helpers

Next, we load:
- `defineInputs.jl` — defines steady-state (`SS`) and time-course (`TC`) input curves for **BCR**, **TLR**, and **NIK**:
  - `bcrSignalSS` (low basal BCR — M-CLL / healthy) vs `bcrSignalSSHigh` (high chronic BCR — U-CLL)
  - `tlrSignalSS` (resting TLR) vs `tlrSignalTC` (acute CpG / TLR stimulus pulse)
  - `nikSignalSS` (basal NIK) vs `nikSignalTC` (non-canonical BAFF/CD40 stimulus)
- `runSimulation.jl` — runs the 2-phase solver (**Phase 1:** 1,000-hour steady state; **Phase 2:** 12-hour dynamic time course using `Rodas4(autodiff=false)`)
- `plotAllSpecies.jl` — plots steady-state levels (left) and 12-hour time courses (right) for any NF-κB species!

In [ ]:
include("defineInputs.jl")

t = 0
if !@isdefined(odeModel)
    include("distributedModelFiles/odeModel.jl")
end
include("variableNames.jl")
include("scanIncludes.jl")

include("runSimulation.jl")
include("plotAllSpecies.jl")

colorArray = palette(:seaborn_colorblind)

first_cell = 1
last_cell  = 1       # Increase (e.g. 5 or 10) to simulate heterogeneous B-cells with mean ± SD error bars & shaded ribbons!
preCV      = 0.11
maxTimeTC  = 12 * 60      # 12-hour dynamic time course (in minutes)
maxTimeSS  = 1000 * 60    # 1,000-hour steady-state equilibration (in minutes)

println("Tri-Model ready with ", length(syms), " molecular species and ", length(parameterNameList), " parameters!")

Tri-Model ready with 140 molecular species and 474 parameters!

## Step 4: Configure the Three CLL Conditions

Look closely at how each experimental condition is set up:
- **`M-CLL CpG`**: Low basal BCR (`bcrSignalSS`) + CpG/TLR time-course pulse (`tlrSignalTC`)
- **`U-CLL CpG`**: High chronic basal BCR (`bcrSignalSSHigh`) + CpG/TLR pulse (`tlrSignalTC`)
- **`U-CLL CpG + drug`**: Same U-CLL inputs, plus an **IKK inhibitor** simulated by increasing `k1_IKKdeactivation-linking` by `10.0×`!

*(Notice `k1_MYD88SELFACTIVATION-TLR` is set to `0.0` here because CLL cells do not have the Waldenström MYD88 L265P mutation—we will switch that on in Chapter 2.3!)*

In [ ]:
conditions     = []
paramsToChange = []
modifyAmount   = []
BCRSSArray     = []
BCRTCArray     = []
TLRSSArray     = []
TLRTCArray     = []
NIKSSArray     = []
NIKTCArray     = []

# 1. M-CLL (low basal BCR + TLR/CpG stimulus)
push!(conditions,     "M-CLL CpG")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR"])
push!(modifyAmount,   [0.0])
push!(BCRSSArray,     bcrSignalSS)
push!(BCRTCArray,     bcrSignalSS)
push!(TLRSSArray,     tlrSignalSS)
push!(TLRTCArray,     tlrSignalTC)
push!(NIKSSArray,     nikSignalSS)
push!(NIKTCArray,     nikSignalSS)

# 2. U-CLL (high chronic basal BCR + TLR/CpG stimulus)
push!(conditions,     "U-CLL CpG")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR"])
push!(modifyAmount,   [0.0])
push!(BCRSSArray,     bcrSignalSSHigh)
push!(BCRTCArray,     bcrSignalSSHigh)
push!(TLRSSArray,     tlrSignalSS)
push!(TLRTCArray,     tlrSignalTC)
push!(NIKSSArray,     nikSignalSS)
push!(NIKTCArray,     nikSignalSS)

# 3. U-CLL + IKK inhibitor (10x faster IKK deactivation)
push!(conditions,     "U-CLL CpG + drug")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR", "k1_IKKdeactivation-linking"])
push!(modifyAmount,   [0.0, 10.0])
push!(BCRSSArray,     bcrSignalSSHigh)
push!(BCRTCArray,     bcrSignalSSHigh)
push!(TLRSSArray,     tlrSignalSS)
push!(TLRTCArray,     tlrSignalTC)
push!(NIKSSArray,     nikSignalSS)
push!(NIKTCArray,     nikSignalSS)

folder = "teachingOutputs"
display(DataFrame(
    Condition  = conditions,
    Parameters = [join(x, ", ") for x in paramsToChange],
    Multiplier = [join(string.(x), ", ") for x in modifyAmount],
))

## Step 5: Run the Simulation & Plot Nuclear NF-κB Dimers

Now run the 2-phase ODE solver for all three conditions (this typically takes ~5–10 seconds for all 140 coupled ODEs!), then plot the three key nuclear NF-κB transcription factor dimers:
- **`RelAnp50n`** — Canonical RelA:p50 in the nucleus
- **`cRelnp50n`** — Canonical cRel:p50 in the nucleus (key driver of B-cell proliferation!)
- **`RelBnp52n`** — Non-canonical RelB:p52 in the nucleus

In [ ]:
runSimulation(
    first_cell, last_cell, conditions, folder,
    BCRSSArray, BCRTCArray, TLRSSArray, TLRTCArray, NIKSSArray, NIKTCArray
)

In [ ]:
hoursToPlot    = 12
colorArray     = palette(:seaborn_colorblind)
speciesToPlot  = ["RelAnp50n", "cRelnp50n", "RelBnp52n"]
maxValOfYAxis  = 0
qualityScaling = 0.5

gr()
plotAllSpecies(
    speciesToPlot, conditions, colorArray,
    first_cell, last_cell, folder,
    hoursToPlot, maxValOfYAxis, qualityScaling, 60
)

## Interpreting Your CLL Results

Look carefully at the **three plots** above (each plot shows **steady-state levels** on the left and the **12-hour post-CpG time course** on the right):

1. How does chronic basal BCR activity in **U-CLL** change both **resting (steady-state)** NF-κB and the **CpG response** compared to **M-CLL**?
2. Does the **IKK inhibitor** (`U-CLL CpG + drug`) suppress `RelAnp50n` and `cRelnp50n`?
3. What happens to the **non-canonical dimer `RelBnp52n`**? Why might canonical IKK inhibition affect (or fail to completely eliminate) non-canonical NF-κB?

````{admonition} 💡 Key Biological Observations
:class: tip, dropdown

- **Elevated basal activity in U-CLL:** Chronic BCR signalling (`bcrSignalSSHigh`) elevates basal canonical NF-κB (`RelAnp50n` and `cRelnp50n`) *and* primes cross-talk with the non-canonical pathway via NF-κB-driven `p100` synthesis (`RelBnp52n`)!
- **IKK inhibition (`k1_IKKdeactivation-linking` × 10):** Strongly suppresses both the steady-state elevation and the CpG-induced spike in canonical **RelA:p50** and **cRel:p50**, bringing U-CLL much closer to M-CLL.
- **Clinical nuance:** Because canonical IKK is active in nearly every cell type in the body, systemic IKK inhibitors often cause toxicity in patients—which is why searching for alternative or pathway-specific drug targets is so important!
````

---

**Ready for the challenge?** Head to [**4.3 Drug Discovery for Waldenström Macroglobulinemia**](03-waldenstrom-drug-discovery.ipynb) to switch on the **MYD88 L265P** mutation and hunt for effective drugs!